# F1 Strategy Analysis

## Loading data

In [ ]:
from pathlib import Path
import json
import logging
import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

DRY = ["SOFT", "MEDIUM", "HARD"]
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import plot_tree

def show_table(frame):
    display(frame)
from sklearn.model_selection import train_test_split


In [ ]:
DATA = Path("data/historical")
fastf1.Cache.enable_cache("data/fastf1_cache")
coverage = pd.read_csv(DATA / "data_audit.csv")

In [ ]:
race_tables = []
race_contexts = {}
excluded_references = []

def has_reference(context):
    reference = context.get("reference_s")
    return (context.get("reference_source") in ["Q", "FP1", "FP2", "FP3"]
            and isinstance(reference, (int, float))
            and np.isfinite(reference) and reference > 0)

for race_id in coverage.loc[coverage["status"] == "included", "race_id"]:
    context = json.loads((DATA / f"{race_id}_context.json").read_text())
    if has_reference(context):
        race_contexts[race_id] = context
    else:
        excluded_references.append(race_id)
    race = pd.read_parquet(DATA / f"{race_id}_R.parquet")
    race["race_id"] = race_id
    race["year"] = context["year"]
    race["event"] = context["event"]
    race["total_laps"] = context["planned_laps"]
    race_tables.append(race)
raw_laps = pd.concat(race_tables, ignore_index=True)
print("Missing references:", excluded_references)

In [ ]:
track_features = pd.DataFrame([
    ["Austin", 5.513, 20, 170, 41, 1000, 160, 0],
    ["Baku", 6.003, 20, -28, 20, 2200, 120, 1],
    ["Barcelona", 4.657, 14, 110, 30, 1050, 170, 0],
    ["Budapest", 4.381, 14, 250, 36, 900, 150, 0],
    ["Imola", 4.909, 19, 40, 30, 1000, 170, 0],
    ["Jeddah", 6.174, 27, 5, 5, 1100, 190, 1],
    ["Las Vegas", 6.201, 17, 610, 10, 1900, 130, 1],
    ["Le Castellet", 5.842, 15, 420, 20, 1100, 160, 0],
    ["Madrid", 5.474, 22, 700, 50, 1000, 150, 1],
    ["Marina Bay", 4.940, 19, 3, 5, 700, 110, 1],
    ["Melbourne", 5.278, 14, 10, 8, 900, 170, 1],
    ["Mexico City", 4.304, 17, 2240, 5, 1200, 140, 0],
    ["Miami", 5.412, 19, 3, 8, 1300, 150, 1],
    ["Monaco", 3.337, 19, 15, 42, 500, 100, 1],
    ["Montréal", 4.361, 14, 12, 8, 1100, 130, 1],
    ["Monza", 5.793, 11, 160, 12, 1100, 170, 0],
    ["Sakhir", 5.412, 15, 10, 17, 1100, 140, 0],
    ["Silverstone", 5.891, 18, 150, 15, 800, 200, 0],
    ["Spa-Francorchamps", 7.004, 19, 400, 102, 1400, 190, 0],
    ["Spielberg", 4.318, 10, 680, 65, 850, 180, 0],
    ["Suzuka", 5.807, 18, 45, 40, 1200, 190, 0],
    ["Yas Island", 5.281, 16, 5, 8, 1200, 140, 0],
    ["Zandvoort", 4.259, 14, 5, 15, 700, 170, 0],
], columns=["circuit", "track_length_km", "turn_count", "altitude_m", "elevation_change_m",
            "longest_straight_m", "average_corner_speed_kmh", "street_circuit"])
show_table(track_features)

In [ ]:
fastf1.set_log_level("ERROR")
logging.getLogger("requests_cache").setLevel(logging.ERROR)

def pre_race_temperatures(context, laps):
    session = fastf1.get_session(context["year"], context["round"], "R")
    session.load(laps=False, telemetry=False, weather=True, messages=False)
    try:
        weather = session.weather_data
    except fastf1.exceptions.DataNotLoadedError:
        return {"expected_air_temp_c": np.nan, "expected_track_temp_c": np.nan}
    before = weather[weather["Time"] <= laps["LapStartTime"].min()]
    reading = before.iloc[-1] if len(before) else weather.iloc[0]
    return {"expected_air_temp_c": reading["AirTemp"], "expected_track_temp_c": reading["TrackTemp"]}

race_temperatures = pd.DataFrame.from_dict({
    race_id: pre_race_temperatures(context, raw_laps[raw_laps["race_id"] == race_id])
    for race_id, context in race_contexts.items()
}, orient="index")

In [ ]:
SUPPLEMENT = Path("data/supplementary_2026")
transfer_contexts = {}
transfer_tables = []
for race_id in ["2026_01", "2026_03", "2026_14"]:
    context = json.loads((SUPPLEMENT / f"{race_id}_context.json").read_text())
    if not has_reference(context):
        print(f"Skipping {race_id}: no usable reference")
        continue
    transfer_contexts[race_id] = context
    race = pd.read_parquet(SUPPLEMENT / f"{race_id}_R.parquet")
    race["race_id"] = race_id
    race["year"] = context["year"]
    race["event"] = context["event"]
    race["total_laps"] = context["planned_laps"]
    transfer_tables.append(race)
transfer_raw = pd.concat(transfer_tables, ignore_index=True)

In [ ]:
transfer_temperatures = pd.DataFrame.from_dict({
    race_id: pre_race_temperatures(context, transfer_raw[transfer_raw["race_id"] == race_id])
    for race_id, context in transfer_contexts.items()
}, orient="index")

## Data cleaning

In [ ]:
raw_laps["lap_seconds"] = raw_laps["LapTime"].dt.total_seconds()
raw_laps["start_seconds"] = raw_laps["LapStartTime"].dt.total_seconds()
raw_laps["end_seconds"] = raw_laps["Time"].dt.total_seconds()

dry_tyres = np.isin(raw_laps["Compound"].to_numpy(), DRY)
green_flag = raw_laps["TrackStatus"].to_numpy() == "1"
no_pit_stop = np.logical_and(raw_laps["PitInTime"].isna().to_numpy(),
                             raw_laps["PitOutTime"].isna().to_numpy())
accurate_timing = raw_laps["IsAccurate"].fillna(False).to_numpy(dtype=bool)
not_generated = np.logical_not(raw_laps["FastF1Generated"].fillna(False).to_numpy(dtype=bool))
timed_lap = np.isfinite(raw_laps["lap_seconds"].to_numpy())
after_start = raw_laps["LapNumber"].to_numpy() > 1
known_tyre_age = np.isfinite(raw_laps["TyreLife"].to_numpy())

In [ ]:
filters = {
    "Dry tyres: soft, medium or hard": dry_tyres,
    "Green flag throughout the lap": green_flag,
    "Neither pit-in nor pit-out lap": no_pit_stop,
    "FastF1 marks timing as accurate": accurate_timing,
    "Not a FastF1-generated lap": not_generated,
    "Recorded lap time": timed_lap,
    "After the opening lap": after_start,
    "Known tyre age": known_tyre_age,
}
keep = np.ones(len(raw_laps), dtype=bool)
cleaning_steps = [{"step": "Raw laps", "removed": 0, "remaining": len(raw_laps)}]
for label, mask in filters.items():
    before = np.count_nonzero(keep)
    keep = np.logical_and(keep, mask)
    remaining = np.count_nonzero(keep)
    cleaning_steps.append({"step": label, "removed": before - remaining, "remaining": remaining})
clean_laps = raw_laps.loc[keep].copy()
show_table(pd.DataFrame(cleaning_steps))

In [ ]:
known_times = np.logical_and(np.isfinite(clean_laps["start_seconds"].to_numpy()),
                             np.isfinite(clean_laps["end_seconds"].to_numpy()))
feature_laps = clean_laps.loc[known_times].copy()
print(f"Laps with timestamps for feature construction: {len(feature_laps):,}")
print(f"Clean laps missing timestamps: {np.count_nonzero(~known_times):,}")

known_position = np.logical_and(np.isfinite(raw_laps["end_seconds"].to_numpy()),
                                np.isfinite(raw_laps["LapNumber"].to_numpy()))
position_laps = raw_laps.loc[known_position].copy()

In [ ]:
def build_race_features(valid, completed_laps, context, race_id):
    reference = context['reference_s']
    finish = context['planned_laps']

    clean_arrays = {}
    for driver, laps in valid.groupby("Driver"):
        laps = laps.sort_values("Time")
        end_times = laps["end_seconds"].to_numpy()
        lap_times = laps["lap_seconds"].to_numpy()
        clean_arrays[driver] = (end_times, lap_times)

    position_arrays = {}
    for driver, laps in completed_laps.groupby("Driver"):
        laps = laps.sort_values("Time")
        end_times = laps["end_seconds"].to_numpy()
        lap_numbers = laps["LapNumber"].to_numpy()
        pit_in = laps["PitInTime"].notna().to_numpy()
        position_arrays[driver] = (end_times, lap_numbers, pit_in)

    rows = []
    for lap in valid.itertuples(index=False):
        start = lap.start_seconds
        recent_paces = {}
        locations = {}
        for driver, (end_times, seconds) in clean_arrays.items():

            k = np.searchsorted(end_times, start, side='right')
            if k:
                recent_paces[driver] = seconds[max(0, k - 3):k].mean()
        if lap.Driver not in recent_paces:
            continue
        pack = np.median(list(recent_paces.values()))
        for driver, (end_times, lap_numbers, pit_in) in position_arrays.items():

            k = np.searchsorted(end_times, start, side='right')
            if not k:
                continue
            elapsed = start - end_times[k - 1]
            pace = recent_paces.get(driver, pack)
            if elapsed > 1.5 * pace or pit_in[k - 1]:
                continue
            locations[driver] = lap_numbers[k - 1] + elapsed / pace
        if lap.Driver not in locations:
            continue
        position = locations[lap.Driver]
        other_positions = np.array([
            location for driver, location in locations.items() if driver != lap.Driver
        ])
        gaps = (other_positions - position) % 1
        gaps = gaps[gaps > 0]
        ahead = gaps * pack
        behind = (1 - gaps) * pack
        row = {
            'race_id': race_id,
            'year': context['year'],
            'circuit': context['circuit'],
            'driver': lap.Driver,
            'lap': lap.LapNumber,
            'start_s': start,
            'end_s': lap.end_seconds,
            'lap_s': lap.lap_seconds,
            'stint': lap.Stint,
            'compound': lap.Compound,
            'reference_s': reference,
            'target': lap.lap_seconds / reference - 1,
            'pack_pace': pack / reference - 1,
            'relative_pace': (recent_paces[lap.Driver] - pack) / reference,
            'pre_race_form': context['pre_race_form'].get(lap.Driver, np.nan),
            'race_progress': lap.LapNumber / finish,
            'tyre_age': lap.TyreLife / finish,
            'tyre_age_sq': (lap.TyreLife / finish) ** 2,
            'soft': int(lap.Compound == 'SOFT'),
            'hard': int(lap.Compound == 'HARD'),
        }
        for threshold in [0.5, 1.0, 2.0, 3.0, 5.0]:
            row[f'ahead_{threshold:.1f}'] = np.count_nonzero(ahead <= threshold)
            row[f'behind_{threshold:.1f}'] = np.count_nonzero(behind <= threshold)
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
feature_tables = []
for race_id, context in race_contexts.items():
    race_clean = feature_laps[feature_laps["race_id"] == race_id]
    race_history = position_laps[position_laps["race_id"] == race_id]
    feature_tables.append(build_race_features(race_clean, race_history, context, race_id))
features = pd.concat(feature_tables, ignore_index=True)

In [ ]:
transfer_raw["lap_seconds"] = transfer_raw["LapTime"].dt.total_seconds()
transfer_raw["start_seconds"] = transfer_raw["LapStartTime"].dt.total_seconds()
transfer_raw["end_seconds"] = transfer_raw["Time"].dt.total_seconds()
transfer_keep = np.logical_and.reduce([
    np.isin(transfer_raw["Compound"], DRY),
    transfer_raw["TrackStatus"].to_numpy() == "1",
    transfer_raw["PitInTime"].isna().to_numpy(),
    transfer_raw["PitOutTime"].isna().to_numpy(),
    transfer_raw["IsAccurate"].fillna(False).to_numpy(dtype=bool),
    ~transfer_raw["FastF1Generated"].fillna(False).to_numpy(dtype=bool),
    np.isfinite(transfer_raw["lap_seconds"]),
    transfer_raw["LapNumber"].to_numpy() > 1,
    np.isfinite(transfer_raw["TyreLife"]),
    np.isfinite(transfer_raw["start_seconds"]),
    np.isfinite(transfer_raw["end_seconds"]),
])
transfer_clean = transfer_raw.loc[transfer_keep].copy()
transfer_positions = transfer_raw.loc[
    np.isfinite(transfer_raw["end_seconds"]) & np.isfinite(transfer_raw["LapNumber"])
].copy()
transfer_features = []
for race_id, context in transfer_contexts.items():
    valid = transfer_clean.loc[transfer_clean["race_id"] == race_id]
    completed = transfer_positions.loc[transfer_positions["race_id"] == race_id]
    transfer_features.append(build_race_features(valid, completed, context, race_id))
transfer_features = pd.concat(transfer_features, ignore_index=True)

In [ ]:
show_table(features.groupby("year").agg(races=("race_id", "nunique"), laps=("lap", "size")))

## Modeling

In [ ]:
development = features[features["year"] < 2025].copy()
train, evaluation = train_test_split(development, test_size=0.30, shuffle=True, random_state=42)
tune, validation = train_test_split(evaluation, test_size=0.50, shuffle=True, random_state=42)
train, tune, validation = train.copy(), tune.copy(), validation.copy()
test = features[features["year"] == 2025].copy()
BASE = ["pack_pace", "pre_race_form", "race_progress", "tyre_age",
        "tyre_age_sq", "soft", "hard"]

def make_linear():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LinearRegression())

def predict_seconds(model, frame, columns):
    return (1 + model.predict(frame[columns])) * frame.reference_s.to_numpy()

def race_mae(frame, prediction):
    return per_race_mae(frame, prediction).mean()

def per_race_mae(frame, predictions):
    errors = abs(frame["lap_s"] - predictions)
    return errors.groupby(frame["race_id"]).mean()

In [ ]:
THRESHOLDS = [0.5, 1.0, 2.0, 3.0, 5.0]
baseline_for_tuning = make_linear()
baseline_for_tuning.fit(train[BASE], train['target'])
tune_effects = tune.copy()
tune_effects['residual_seconds'] = (
    tune['lap_s'] - predict_seconds(baseline_for_tuning, tune, BASE)
)

def within_stint_slopes(frame, columns):
    keys = ['race_id', 'driver', 'stint', 'compound']
    values = columns + ['residual_seconds']
    centered = frame[values] - frame.groupby(keys)[values].transform('mean')
    centered['race_id'] = frame['race_id']
    rows = {}
    for race_id, race in centered.groupby('race_id'):
        fit = LinearRegression(fit_intercept=False)
        fit.fit(race[columns], race['residual_seconds'])
        rows[race_id] = fit.coef_
    return pd.DataFrame.from_dict(rows, orient='index', columns=columns)

In [ ]:
def learn_threshold(prefix):
    rows = []
    for threshold in THRESHOLDS:
        column = f'{prefix}_{threshold:.1f}'
        columns = BASE + [column]
        candidate = make_linear()
        candidate.fit(train[columns], train['target'])
        predictions = predict_seconds(candidate, tune, columns)
        slopes = within_stint_slopes(tune_effects, [column])
        rows.append({'threshold_seconds': threshold,
                     'mean_race_mae': race_mae(tune, predictions),
                     'residual_seconds_per_car': slopes[column].mean()})
    return pd.DataFrame(rows)

attack_search = learn_threshold('ahead')
BEST_ATTACK = float(attack_search.loc[
    attack_search['mean_race_mae'].idxmin(), 'threshold_seconds'])
show_table(attack_search.round(4))

In [ ]:
defend_search = learn_threshold('behind')
BEST_DEFEND = float(defend_search.loc[
    defend_search['mean_race_mae'].idxmin(), 'threshold_seconds'])
show_table(defend_search.round(4))

In [ ]:
for frame in [features, train, tune, validation, test]:
    frame['attacking'] = frame[f'ahead_{BEST_ATTACK:.1f}']
    frame['defending'] = frame[f'behind_{BEST_DEFEND:.1f}']
EXTRA = ['attacking', 'defending', 'relative_pace']
feature_definitions = dict(zip(['Attacking', 'Defending', 'Relative pace'], EXTRA))
development_fit = features[features["year"].isin([2022, 2023, 2024])].copy()
confirmation_baseline = make_linear()
confirmation_baseline.fit(development_fit[BASE], development_fit['target'])
base_predictions = predict_seconds(confirmation_baseline, test, BASE)
base_race_errors = per_race_mae(test, base_predictions)

In [ ]:
columns = BASE + EXTRA

validation_training = pd.concat([train, tune])
linear_validation = make_linear()
linear_validation.fit(validation_training[columns], validation_training["target"])
validation_prediction = predict_seconds(linear_validation, validation, columns)
print(f"Validation linear MAE: {race_mae(validation, validation_prediction):.3f} seconds.")

linear_models = {}
for label, inputs in {"Baseline": BASE, "Engineered": columns}.items():
    model = make_linear()
    model.fit(development_fit[inputs], development_fit["target"])
    linear_models[label] = model

In [ ]:
CIRCUIT_COLUMNS = ["track_length_km", "turn_count", "altitude_m", "elevation_change_m",
                   "longest_straight_m", "average_corner_speed_kmh"]
TEMPERATURE_COLUMNS = ["expected_air_temp_c", "expected_track_temp_c"]
continuous_track_columns = CIRCUIT_COLUMNS + TEMPERATURE_COLUMNS
TRACK = continuous_track_columns + ["street_circuit"]
circuit_values = track_features.set_index("circuit")

def attach_track_features(frame, temperatures):
    for column in circuit_values:
        frame[column] = frame["circuit"].map(circuit_values[column])
    for column in temperatures:
        frame[column] = frame["race_id"].map(temperatures[column])

model_frames = [development_fit, validation_training, validation, test]
for frame in model_frames:
    attach_track_features(frame, race_temperatures)

In [ ]:
track_columns = columns + TRACK
track_validation = make_linear()
track_validation.fit(validation_training[track_columns], validation_training["target"])
track_validation_prediction = predict_seconds(track_validation, validation, track_columns)
print(f"Validation linear MAE with track features: {race_mae(validation, track_validation_prediction):.3f} seconds.")

track_model = make_linear()
track_model.fit(development_fit[track_columns], development_fit["target"])

In [ ]:
transfer_features["attacking"] = transfer_features[f"ahead_{BEST_ATTACK:.1f}"]
transfer_features["defending"] = transfer_features[f"behind_{BEST_DEFEND:.1f}"]
attach_track_features(transfer_features, transfer_temperatures)

In [ ]:
CHECKPOINTS = [10, 20, 30]
WINDOW_LENGTH = 10

In [ ]:
def observed_first_stop(driver_laps, checkpoint):
    future = driver_laps.loc[driver_laps.LapNumber.gt(checkpoint)].sort_values("LapNumber")
    stops = future.loc[future.PitInTime.notna()]
    if stops.empty:
        return "No later stop observed", np.nan
    stop = int(stops.iloc[0].LapNumber)
    returned = future.loc[future.LapNumber.eq(stop + 1)]
    if returned.empty or returned.PitOutTime.isna().all():
        return "No confirmed return from pit", stop
    interval = future.loc[future.LapNumber.le(stop)]
    if set(interval.LapNumber) != set(range(checkpoint + 1, stop + 1)):
        return "Incomplete timing before stop", stop
    if not interval.Compound.isin(DRY).all() or not interval.TrackStatus.eq("1").all():
        return "Wet tyres or non-green flags before stop", stop
    return "Scored", stop

def historical_stops(training):
    laps = raw_laps[raw_laps["race_id"].isin(training["race_id"])]
    stops = laps[laps["PitInTime"].notna()].sort_values("LapNumber")
    stops = stops.groupby(["race_id", "Driver"]).head(1).copy()
    stops["stop_fraction"] = stops["LapNumber"] / stops["total_laps"]
    return stops

In [ ]:
def build_team_decisions(raw_laps, features):
    rows = []
    pit_laps = raw_laps[raw_laps["PitInTime"].notna()]
    all_first_stops = pit_laps.groupby(["race_id", "Driver"])["Time"].min()
    field_sizes = raw_laps.groupby("race_id")["Driver"].nunique()
    driver_tables = {key: table for key, table in features.groupby(["race_id", "driver"])}
    for (race_id, driver), laps in raw_laps.groupby(["race_id", "Driver"]):
        laps = laps.sort_values("LapNumber")
        driver_features = driver_tables.get((race_id, driver), features.iloc[:0])
        if race_id in all_first_stops.index.get_level_values(0):
            rival_stops = all_first_stops.loc[race_id].drop(driver, errors="ignore")
        else:
            rival_stops = pd.Series(dtype="timedelta64[ns]")
        field_size = field_sizes[race_id]
        for checkpoint in CHECKPOINTS:
            observed = laps[laps["LapNumber"] <= checkpoint]
            current = driver_features[driver_features["lap"] == checkpoint]
            if observed["PitInTime"].notna().any() or current.empty:
                continue
            last = observed[observed["LapNumber"] == checkpoint].iloc[0]
            snapshot = current.iloc[0]
            checkpoint_time = last["end_seconds"]
            history = driver_features[driver_features["end_s"] <= checkpoint_time].sort_values("end_s")
            recent = history.tail(3)
            reference = snapshot["reference_s"]
            recent_seconds = recent["lap_s"].to_numpy()
            finish = int(last["total_laps"])
            if finish - checkpoint < 11:
                continue
            status, actual = observed_first_stop(laps, checkpoint)
            rows.append({"race_id": race_id, "driver": driver, "checkpoint": checkpoint,
                "year": int(last["year"]), "finish": finish, "compound": last["Compound"],
                "actual_stop": actual, "status": status,
                "race_progress": checkpoint / finish, "current_age": last["TyreLife"] / finish,
                "soft": int(last["Compound"] == "SOFT"), "hard": int(last["Compound"] == "HARD"),
                "position": last["Position"] / field_size,
                "recent_pace": np.mean(recent_seconds) / reference - 1,
                "pace_trend": (recent_seconds[-1] - recent_seconds[0]) / reference,
                "attacking": snapshot["attacking"], "defending": snapshot["defending"],
                "rivals_stopped": (rival_stops <= last["Time"]).sum() / max(field_size - 1, 1)})
    return pd.DataFrame(rows)

In [ ]:
team_decisions = build_team_decisions(raw_laps, features)

In [ ]:
CURRENT_INPUTS = ["race_progress", "current_age", "soft", "hard", "position",
                  "recent_pace", "pace_trend", "attacking", "defending", "rivals_stopped"]
WINDOW_INPUTS = ["window_progress", "window_progress_sq", "window_x_age",
                 "window_x_soft", "window_x_hard", "window_x_pace", "window_x_rivals"]
TEAM_INPUTS = CURRENT_INPUTS + WINDOW_INPUTS

def make_window_rows(decisions):
    tables = []
    for decision_id, row in decisions.iterrows():
        starts = np.arange(int(row["checkpoint"]) + 1, int(row["finish"]) - WINDOW_LENGTH + 1)
        windows = pd.DataFrame([row[CURRENT_INPUTS].to_dict()] * len(starts))
        windows["decision_id"] = decision_id
        windows["window_start"] = starts
        windows["window_end"] = starts + WINDOW_LENGTH - 1
        midpoint = (starts + (WINDOW_LENGTH - 1) / 2) / row["finish"]
        windows["window_progress"] = midpoint
        windows["window_progress_sq"] = midpoint ** 2
        for suffix, feature in [("age", "current_age"), ("soft", "soft"), ("hard", "hard"),
                                ("pace", "recent_pace"), ("rivals", "rivals_stopped")]:
            windows[f"window_x_{suffix}"] = midpoint * row[feature]
        tables.append(windows)
    return pd.concat(tables, ignore_index=True)

In [ ]:
eligible_decisions = team_decisions[team_decisions["status"] == "Scored"]
team_windows = make_window_rows(eligible_decisions)
actual = team_windows["decision_id"].map(eligible_decisions["actual_stop"])
team_windows["hit"] = actual.between(team_windows["window_start"], team_windows["window_end"]).astype(int)
team_windows["race_id"] = team_windows["decision_id"].map(eligible_decisions["race_id"])

def window_training(race_ids):
    return team_windows[team_windows["race_id"].isin(race_ids)]

In [ ]:
def make_forest(depth):
    return make_pipeline(SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=150, max_depth=depth,
                               min_samples_leaf=40, random_state=42, n_jobs=-1))

def score_team_windows(model, decisions):
    candidates = make_window_rows(decisions)
    candidates["probability"] = model.predict_proba(candidates[TEAM_INPUTS])[:, 1]
    best = candidates.loc[candidates.groupby("decision_id")["probability"].idxmax()]
    results = decisions.loc[best["decision_id"], ["race_id", "driver", "checkpoint", "actual_stop"]].copy()
    results["window_start"] = best["window_start"].to_numpy()
    results["window_end"] = best["window_end"].to_numpy()
    results["accuracy_pct"] = 100 * results["actual_stop"].between(results["window_start"], results["window_end"])
    return results

def mean_window_accuracy(results):
    return results.groupby("race_id")["accuracy_pct"].mean().mean()

In [ ]:
def score_historical_windows(training, decisions):
    stops = historical_stops(training)
    rows = []
    for decision_id, row in decisions.iterrows():
        past = stops[(stops["Compound"] == row["compound"]) &
                     (stops["stop_fraction"] > row["race_progress"])]

        if past.empty:
            past = stops[stops["stop_fraction"] > row["race_progress"]]
        candidates = make_window_rows(decisions.loc[[decision_id]])
        stop_laps = np.rint(past["stop_fraction"] * row["finish"])
        counts = [stop_laps.between(w["window_start"], w["window_end"]).sum()
                  for _, w in candidates.iterrows()]
        best = candidates.iloc[int(np.argmax(counts))]
        rows.append({"race_id": row["race_id"], "driver": row["driver"],
            "checkpoint": row["checkpoint"], "actual_stop": row["actual_stop"],
            "window_start": best["window_start"], "window_end": best["window_end"],
            "accuracy_pct": 100 * (best["window_start"] <= row["actual_stop"] <= best["window_end"])})
    return pd.DataFrame(rows)

In [ ]:
development_races = sorted(development_fit["race_id"].unique())
pit_train_races, pit_evaluation_races = train_test_split(
    development_races, test_size=0.30, shuffle=True, random_state=42)
pit_tune_races, pit_validation_races = train_test_split(
    pit_evaluation_races, test_size=0.50, shuffle=True, random_state=42)
training_windows = window_training(pit_train_races)
tuning_decisions = eligible_decisions[eligible_decisions["race_id"].isin(pit_tune_races)]
forest_tuning = []
for depth in [4, 8]:
    model = make_forest(depth)
    model.fit(training_windows[TEAM_INPUTS], training_windows["hit"])
    accuracy = mean_window_accuracy(score_team_windows(model, tuning_decisions))
    forest_tuning.append({"depth": depth, "tuning_accuracy_pct": accuracy})
forest_tuning = pd.DataFrame(forest_tuning)
best_depth = int(forest_tuning.loc[forest_tuning["tuning_accuracy_pct"].idxmax(), "depth"])
show_table(forest_tuning.round(2))

In [ ]:
development_windows = window_training(pit_train_races + pit_tune_races)
forest_validation = make_forest(best_depth)
forest_validation.fit(development_windows[TEAM_INPUTS], development_windows["hit"])
validation_decisions = eligible_decisions[eligible_decisions["race_id"].isin(pit_validation_races)]
print(f"Validation window accuracy: {mean_window_accuracy(score_team_windows(forest_validation, validation_decisions)):.1f}%")

In [ ]:
final_windows = window_training(development_fit["race_id"])
forest = make_forest(best_depth)
forest.fit(final_windows[TEAM_INPUTS], final_windows["hit"])
test_decisions = eligible_decisions[eligible_decisions["race_id"].isin(test["race_id"])]
forest_predictions = score_team_windows(forest, test_decisions)
historical_predictions = score_historical_windows(development_fit, test_decisions)

In [ ]:
other_lap_scores = []
for name in ["Gradient boosting", "Small neural network"]:
    for extra_inputs in [False, True]:
        inputs = track_columns if extra_inputs else BASE + EXTRA
        if name == "Gradient boosting":
            model = make_pipeline(SimpleImputer(strategy="median"),
                HistGradientBoostingRegressor(max_iter=150, max_leaf_nodes=15,
                                              learning_rate=.05, random_state=42))
        else:
            model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                MLPRegressor(hidden_layer_sizes=(16, 8), max_iter=300,
                             early_stopping=True, random_state=42))
        model.fit(development_fit[inputs], development_fit["target"])
        predicted = predict_seconds(model, test, inputs)
        errors = abs(test["lap_s"] - predicted)
        within = (100 * (errors <= 1)).groupby(test["race_id"]).mean().mean()
        other_lap_scores.append({"model": name, "Circuit and weather": extra_inputs,
            "MAE (s)": race_mae(test, predicted), "Within 1 second (%)": within})

## Testing

In [ ]:
race_gains = {}
for label, feature in feature_definitions.items():
    feature_columns = BASE + [feature]
    candidate = make_linear()
    candidate.fit(development_fit[feature_columns], development_fit['target'])
    predictions = predict_seconds(candidate, test, feature_columns)
    race_gains[label] = base_race_errors - per_race_mae(test, predictions)
race_gains = pd.DataFrame(race_gains)

In [ ]:
def summarize_race_tests(race_values):
    rows = []
    for label in race_values:
        values = race_values[label].dropna()
        interval = stats.t.interval(0.95, len(values) - 1,
                                   loc=values.mean(), scale=stats.sem(values))
        rows.append({'predictor': label, 'races': len(values),
                     'estimate': values.mean(), 'ci_low': interval[0],
                     'ci_high': interval[1],
                     'p_value': stats.ttest_1samp(values, 0).pvalue})
    results = pd.DataFrame(rows).set_index('predictor')
    return results

predictive_tests = summarize_race_tests(race_gains)
show_table(predictive_tests[["estimate", "ci_low", "ci_high", "p_value"]].round(5))

In [ ]:
association_data = test.copy()
association_data['residual_seconds'] = test['lap_s'] - base_predictions
association_data['relative_seconds'] = (
    association_data['relative_pace'] * association_data['reference_s']
)
association_columns = EXTRA[:2] + ['relative_seconds']
race_associations = within_stint_slopes(association_data, association_columns)
race_associations.columns = list(feature_definitions)
association_tests = summarize_race_tests(race_associations)
show_table(association_tests[["estimate", "ci_low", "ci_high", "p_value"]].round(5))

In [ ]:
lap_results = []
predictions_by_model = {
    "Baseline": predict_seconds(linear_models["Baseline"], test, BASE),
    "Engineered": predict_seconds(linear_models["Engineered"], test, columns),
    "Recent pace": (1 + test["pack_pace"] + test["relative_pace"]) * test["reference_s"],
}
for label, predicted in predictions_by_model.items():
    errors = test[["race_id", "driver", "lap", "lap_s"]].copy()
    errors["prediction_s"] = np.asarray(predicted)
    errors["error_s"] = abs(errors["lap_s"] - errors["prediction_s"])
    errors["within_1s_pct"] = 100 * (errors["error_s"] <= 1)
    by_race = errors.groupby("race_id")[["error_s", "within_1s_pct"]].mean()
    interval = stats.t.interval(.95, len(by_race) - 1, loc=by_race["error_s"].mean(),
                               scale=stats.sem(by_race["error_s"]))
    lap_results.append({"model": label, "MAE (s)": by_race["error_s"].mean(),
        "MAE CI low": interval[0], "MAE CI high": interval[1],
        "Within 1 second (%)": by_race["within_1s_pct"].mean(), "races": len(by_race), "laps": len(errors)})

In [ ]:
lap_results = pd.DataFrame(lap_results)
show_table(lap_results.round(3))

In [ ]:
track_predictions = {
    "Engineered linear": predict_seconds(linear_models["Engineered"], test, columns),
    "Engineered linear + track features": predict_seconds(track_model, test, track_columns),
}
track_results = []
track_race_errors = {}
for label, predicted in track_predictions.items():
    errors = pd.DataFrame({"error_s": np.abs(test["lap_s"].to_numpy() - predicted)})
    errors["within_1s_pct"] = 100 * (errors["error_s"] <= 1)
    by_race = errors.groupby(test["race_id"].to_numpy()).mean()
    interval = stats.t.interval(.95, len(by_race) - 1, loc=by_race["error_s"].mean(),
                               scale=stats.sem(by_race["error_s"]))
    track_results.append({"model": label, "MAE (s)": by_race["error_s"].mean(),
        "MAE CI low": interval[0], "MAE CI high": interval[1],
        "Within 1 second (%)": by_race["within_1s_pct"].mean(), "races": len(by_race), "laps": len(errors)})
    track_race_errors[label] = by_race["error_s"]
track_results = pd.DataFrame(track_results)
track_race_errors = pd.DataFrame(track_race_errors)
show_table(track_results.round(3))

In [ ]:
mae_without_track = track_race_errors["Engineered linear"]
mae_with_track = track_race_errors["Engineered linear + track features"]
gain = mae_without_track - mae_with_track
track_test = summarize_race_tests(pd.DataFrame({"Track features": gain}))
show_table(track_test[["races", "estimate", "ci_low", "ci_high", "p_value"]].round(4))

In [ ]:
pit_results = pd.concat([forest_predictions.assign(model="Random forest"),
                         historical_predictions.assign(model="Historical baseline")], ignore_index=True)
pit_by_race = pit_results.groupby(["race_id", "model"])["accuracy_pct"].mean().unstack()
pit_summary = []
for name in pit_by_race:
    values = pit_by_race[name]
    interval = stats.t.interval(.95, len(values) - 1, loc=values.mean(), scale=stats.sem(values))
    pit_summary.append({"model": name, "accuracy_pct": values.mean(),
        "CI low": interval[0], "CI high": interval[1], "races": len(values), "checkpoints": len(test_decisions)})
pit_summary = pd.DataFrame(pit_summary)
show_table(pit_summary.round(2))
show_table(summarize_race_tests(pd.DataFrame({"Forest gain (percentage points)":
    pit_by_race["Random forest"] - pit_by_race["Historical baseline"]})).round(4))

In [ ]:
show_table(pd.DataFrame(other_lap_scores).round(3))

In [ ]:
transfer_predictions = {
    "Basic linear": predict_seconds(linear_models["Baseline"], transfer_features, BASE),
    "Engineered linear": predict_seconds(linear_models["Engineered"], transfer_features, BASE + EXTRA),
    "Engineered linear + track features": predict_seconds(track_model, transfer_features, track_columns),
}
transfer_lap_tables = []
for model_name, predictions in transfer_predictions.items():
    errors = transfer_features[["race_id"]].copy()
    errors["absolute_error"] = np.abs(transfer_features["lap_s"].to_numpy() - np.asarray(predictions))
    errors["within_1s"] = 100 * (errors["absolute_error"] <= 1)
    summary = errors.groupby("race_id").agg(
        laps=("absolute_error", "size"),
        mae_s=("absolute_error", "mean"),
        within_1s_pct=("within_1s", "mean"),
    ).reset_index()
    summary["model"] = model_name
    transfer_lap_tables.append(summary)
transfer_lap_results = pd.concat(transfer_lap_tables, ignore_index=True)
transfer_names = {key: value["circuit"] for key, value in transfer_contexts.items()}
transfer_lap_results["circuit"] = transfer_lap_results["race_id"].map(transfer_names)
transfer_lap_results = transfer_lap_results[
    (transfer_lap_results["model"] != "Basic linear") |
    (transfer_lap_results["circuit"] == "Melbourne")
]
show_table(transfer_lap_results[["circuit", "model", "laps", "mae_s", "within_1s_pct"]].round(3))

In [ ]:
bahrain_raw = raw_laps[raw_laps["race_id"] == "2024_01"].copy()
bahrain_laps = np.arange(1, int(bahrain_raw["LapNumber"].max()) + 1)
sector_times = bahrain_raw.loc[
    bahrain_raw["IsAccurate"].fillna(False)
    & ~bahrain_raw["FastF1Generated"].fillna(False)
    & bahrain_raw["PitInTime"].isna()
    & bahrain_raw["PitOutTime"].isna()
    & bahrain_raw["LapNumber"].gt(1)
    & bahrain_raw["Compound"].isin(["SOFT", "MEDIUM", "HARD"])
    & bahrain_raw["LapTime"].notna()
].copy()
bahrain_laps = np.arange(2, int(bahrain_raw["LapNumber"].max()) + 1)
timing_columns = {"Sector 1": "Sector1Time", "Sector 2": "Sector2Time",
                  "Sector 3": "Sector3Time", "Full lap": "LapTime"}
for label, column in timing_columns.items():
    sector_times[label] = sector_times[column].dt.total_seconds()

In [ ]:
timing_summary = []
for label in timing_columns:
    early = sector_times[sector_times["LapNumber"].between(2, 11)].groupby("Driver")[label].mean()
    late = sector_times[sector_times["LapNumber"].between(48, 57)].groupby("Driver")[label].mean()
    paired = pd.concat([early.rename("early"), late.rename("late")], axis=1).dropna()
    timing_summary.append({"Timing": label, "Drivers": len(paired),
                           "Early mean (s)": paired["early"].mean(),
                           "Late mean (s)": paired["late"].mean(),
                           "Late minus early (s)": (paired["late"] - paired["early"]).mean()})
show_table(pd.DataFrame(timing_summary).round(3))

## Plotting

In [ ]:
bahrain_scatter = bahrain_raw.dropna(subset=["LapTime", "LapNumber", "Stint"]).copy()
bahrain_scatter["seconds"] = bahrain_scatter["LapTime"].dt.total_seconds()
colors = {1: "#2878B5", 2: "#E58B23", 3: "#389B80", 4: "#985EA8"}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True,
                         gridspec_kw={"width_ratios": [1, 1.7]})
for stint, laps in bahrain_scatter.groupby("Stint"):
    for ax in axes:
        ax.scatter(laps["LapNumber"], laps["seconds"], s=17, alpha=.55,
                   color=colors[int(stint)], edgecolors="none", label=f"Stint {int(stint)}")
for ax in axes:
    ax.set(xlabel="Race lap", xlim=(.5, 57.5))
    ax.grid(axis="y", alpha=.18)
axes[0].set(title="All recorded lap times", ylabel="Lap time (seconds)")
axes[1].set(title="Same points · closer view of 92–102 seconds", ylim=(92, 102))
axes[1].legend(title="Driver's stint number", fontsize=9, loc="upper right")
fig.suptitle("Bahrain 2024 · lap times combine fuel, tyre and race-context effects", fontsize=14)
fig.tight_layout(rect=[0,0,1,.94])
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
colors = ["#2878B5", "#D6792D", "#389B80", "#6554A4"]
for ax, label, color in zip(axes.flat, timing_columns, colors):
    by_lap = sector_times.groupby("LapNumber")[label]
    mean = by_lap.mean().reindex(bahrain_laps)
    lower = by_lap.quantile(.25).reindex(bahrain_laps)
    upper = by_lap.quantile(.75).reindex(bahrain_laps)
    ax.fill_between(bahrain_laps, lower, upper, color=color, alpha=.18,
                    label="Middle 50% of drivers")
    ax.plot(bahrain_laps, mean, color=color, linewidth=2, label="Field mean")
    ax.set(title=label, ylabel="Time (seconds)", xlim=(2, 57))
    ax.grid(axis="y", alpha=.2)
for ax in axes[1]:
    ax.set_xlabel("Race lap")
axes[0, 0].legend(fontsize=9)
fig.suptitle("Bahrain 2024 · sector and lap times outside pit stops", fontsize=15)
fig.tight_layout(rect=[0, 0, 1, .95])

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7), sharex=True, sharey=True)
x = np.linspace(tune.lap_s.min() - 2, tune.lap_s.max() + 2, 700)
colors = ["#526D82", "#D6792D", "#32836C"]
for ax, column, title in zip(axes, ["ahead_3.0", "behind_3.0"], ["Cars ahead", "Cars behind"]):
    groups = np.minimum(tune[column].to_numpy(), 2).astype(int)
    for group, color in enumerate(colors):
        values = tune.loc[groups == group, "lap_s"].to_numpy()
        label = ("2+ cars" if group == 2 else f"{group} car" + ("s" if group == 0 else ""))
        ax.plot(x, stats.gaussian_kde(values)(x), color=color, lw=2,
                label=f"{label} (n={len(values):,})")
    ax.set(title=title, xlabel="Actual lap time (seconds)")
    ax.legend(frameon=False, fontsize=9)
    ax.grid(axis="y", alpha=.15)
axes[0].set_ylabel("Density")
fig.suptitle("Actual lap times by nearby-car count · three-second gap")
fig.tight_layout()
plt.show()

In [ ]:
def plot_threshold_search(results, label):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    x = results['threshold_seconds']
    axes[0].plot(x, results['mean_race_mae'], marker='o')
    axes[0].set_ylabel('Mean race MAE (seconds)')
    axes[0].set_title('Prediction: lower is better')
    axes[1].plot(x, results['residual_seconds_per_car'], marker='o')
    axes[1].axhline(0, color='gray', linestyle='--')
    axes[1].set_ylabel('Baseline error change (seconds per car)')
    axes[1].set_title('Within-stint association; exploratory')
    for ax in axes:
        ax.set_xlabel('Distance threshold (seconds)')
    fig.suptitle(f'{label}: shuffled development tuning laps')
    fig.tight_layout()
    plt.show()

plot_threshold_search(attack_search, 'Attacking')

In [ ]:
plot_threshold_search(defend_search, 'Defending')

In [ ]:
bahrain = features.loc[features.race_id.eq("2024_01")].copy()
bahrain["recent_pace_seconds"] = bahrain.relative_pace * bahrain.reference_s
fig, ax = plt.subplots(figsize=(9, 4.2))
for compound, label, color in [("SOFT", "Soft", "#E53935"),
                                ("MEDIUM", "Medium", "#F2C500"),
                                ("HARD", "Hard", "#808080")]:
    points = bahrain.loc[bahrain.compound.eq(compound)]
    ax.scatter(points.lap_s, points.recent_pace_seconds, s=18,
               alpha=.65, color=color, edgecolors="none", label=label,
               rasterized=True)
ax.legend(title="Tire compound", loc="lower right", framealpha=.9,
          fontsize=9, title_fontsize=9, markerscale=1.5)
ax.axhline(0, color="#777777", linestyle="--", linewidth=1)
ax.set(xlabel="Actual current lap time (seconds)",
       ylabel="Recent pace minus field median (seconds)",
       title="Bahrain 2024: recent relative pace and the next observed lap")
ax.text(.02, .97, "Above zero: recently slower than the field\nBelow zero: recently faster than the field",
        transform=ax.transAxes, va="top", fontsize=10,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": .85})
ax.grid(alpha=.13)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, results, title, xlabel in [
    (axes[0], predictive_tests, '2025 prediction gain',
     'MAE reduction (seconds); positive is better'),
    (axes[1], association_tests, 'Within-stint association with baseline error',
     'Seconds / car, or seconds / second of relative pace')
]:
    positions = np.arange(len(results))
    ax.hlines(positions, results['ci_low'], results['ci_high'], color='steelblue')
    ax.scatter(results['estimate'], positions, color='steelblue')
    ax.axvline(0, color='gray', linestyle='--')
    ax.set_yticks(positions, results.index)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
fig.suptitle('2025 test races: mean and individual 95% confidence interval')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
lap_results.set_index("model")["MAE (s)"].plot.barh(ax=axes[0])
axes[0].set(xlabel="2025 mean absolute error (s)", ylabel="")

coefficients = pd.Series(linear_models["Engineered"].named_steps["linearregression"].coef_, index=columns)
(100 * coefficients).sort_values().plot.barh(ax=axes[1])
axes[1].set(xlabel="Seconds per training SD on a 100-second reference lap", ylabel="")
plt.tight_layout()
plt.show()

In [ ]:
laps = np.arange(1, 61)
pit_laps = np.arange(10, 46)
def race_time(pit_lap):
    medium_age = laps - 1
    hard_age = np.maximum(laps - pit_lap - 1, 0)
    medium = .065 * medium_age + .010 * np.maximum(medium_age - 20, 0) ** 2
    hard = .65 + .045 * hard_age
    lap_time = 92 - .035 * (laps - 1) + np.where(laps <= pit_lap, medium, hard)
    return float(lap_time.sum() + 22)

times = np.array([race_time(p) for p in pit_laps])
loss = times - times.min()
best = int(pit_laps[np.argmin(times)])
near = pit_laps[loss <= 2]
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
fig, ax = plt.subplots(figsize=(10.5, 4.5))
ax.axvspan(near.min()-.5, near.max()+.5, color='#c6e5d5', alpha=.65,
           label=f'Within 2 seconds of best: laps {near.min()}–{near.max()}')
ax.plot(pit_laps, loss, color='#526D82', linewidth=2.5)
ax.scatter([best],[0],color='#D6792D',s=70,zorder=5)
ax.annotate(f'Best simulated stop: end of lap {best}', (best,0), xytext=(best-2,32),
            arrowprops={'arrowstyle':'->','color':'#D6792D'}, color='#9e511a', fontsize=11)
ax.text(12, loss[2]+4,'Too early\nLonger second stint',color='#526D82')
ax.text(34, 46,'Too late\nMore time on worn mediums',color='#526D82')
ax.set(xlabel='Pit at the end of race lap', ylabel='Total race time lost versus best stop (seconds)',
       title='Simulated pit window: when does switching tyres save the most time?',
       xlim=(10,45),ylim=(-2,float(loss.max()+12)))
ax.grid(axis='y',alpha=.18)
ax.legend(loc='upper center',frameon=False,fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
checkpoint_hits = pit_results.pivot(index=["model", "race_id", "driver"],
                                    columns="checkpoint", values="accuracy_pct")
checkpoint_hits = checkpoint_hits.reindex(columns=[10, 20, 30]).dropna()
checkpoint_means = checkpoint_hits.groupby(["model", "race_id"]).mean()
checkpoint_means = checkpoint_means.groupby("model").mean()

race_names = raw_laps.drop_duplicates("race_id").set_index("race_id")["event"]
plot_races = pit_by_race.rename(index=race_names.to_dict())
fig, axes = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [1.2, 1]})
plot_races.plot.barh(ax=axes[0], color=["#526D82", "#D6792D"])
axes[0].set(xlabel="Actual stop inside 10-lap window (%)", ylabel="",
            title="2025 accuracy by race", xlim=(0, 100))
axes[0].legend(title="", fontsize=8, loc="lower right")
for model, color in [("Historical baseline", "#526D82"), ("Random forest", "#D6792D")]:
    axes[1].plot([10, 20, 30], checkpoint_means.loc[model], "o-", color=color,
                 linewidth=2, markersize=7, label=model)
axes[1].set(xlabel="Forecast made after race lap", ylabel="Actual stop inside 10-lap window (%)",
            title="Do forecasts improve later in the race?", xticks=[10, 20, 30], ylim=(0, 100))
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", alpha=.2)
fig.tight_layout(rect=[0,0,1,1])
plt.show()

In [ ]:
forest_estimator = forest.named_steps["randomforestclassifier"]
fig, ax = plt.subplots(figsize=(15, 5.2))
plot_tree(forest_estimator.estimators_[0], max_depth=2,
          feature_names=TEAM_INPUTS, class_names=["Outside window", "Inside window"],
          filled=True, rounded=True, impurity=False, proportion=True,
          precision=2, fontsize=9, ax=ax)
ax.set_title("One fitted tree from the 150-tree random forest")
fig.tight_layout()
plt.show()